# Chapter 27
## Phase Locking with Delays
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter27.ipynb)

## About this chapter

Propagation delays change the phase at which an oscillator receives a pulse,
so they change the locking map itself. A delayed pulse arrives after its
source phase has already advanced, and this extra timing must be folded in
before applying a phase response curve. As a result a delay can stabilize a
phase difference that was not locked without delay, or destabilize
synchronous timing that was stable without it.

The examples below simulate delayed pulse coupling for two and three
event-driven phase oscillators, then realize the same kind of delayed
locking with a pair of continuous theta neurons. For natural period $T$ and
transmission delay $d$, the event map applies a phase shift at a phase
displaced by $d/T$; in the theta-neuron realization the continuous angle
keeps evolving between discrete pulse arrivals, so the delay is carried as
an event-time condition rather than folded into a static term.

See [`chapter27.md`](chapter27.md) for the full guide, including suggested
order and related chapters.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon


## Two Delayed Pulse-Coupled Oscillators

Two phase oscillators A and B exchange pulses that arrive a fixed delay
$\delta$ (in units of the intrinsic period $T$) after they are sent. Each
event -- a self-spike (phase reset) or a delayed pulse arrival (a jump
$\varphi\to\varphi+g(\varphi)$) -- is processed exactly as it occurs, so the
simulation advances event-to-event rather than on a fixed time grid.
`simulate_two_delayed_pulse_coupled_osc` runs the pair for a short delay
($\delta=0.1$) and a long delay ($\delta=0.7$) and returns both runs' spike
times.


In [ ]:
def two_pulse_g(phi, epsilon=1 / 3):
    return epsilon * phi * (1 - phi)


def two_pulse_f(phi, epsilon=1 / 3):
    return phi + two_pulse_g(phi, epsilon)


def simulate_two_delayed_pulse_pair(delta, t_final=20., epsilon=1 / 3):
    '''event-driven simulation of two pulse-coupled oscillators A, B with
    a transmission delay delta (in units of the intrinsic period): each
    spike's effect on the other oscillator arrives delta time units
    later, rather than instantaneously.'''
    phi_A, phi_B = 0., 0.9
    t_A_to_B = delta  # time until the next A-input reaches B
    t_B_to_A = np.inf  # time until the next B-input reaches A
    t_present = 0.

    num_spikes_A, num_spikes_B = 1, 0
    t_spikes_A = [0.]
    t_spikes_B = []

    while t_present < t_final:
        T_vec = [1 - phi_A, 1 - phi_B, t_A_to_B, t_B_to_A]
        T_0 = min(T_vec)
        done = False

        if T_0 == 1 - phi_A:
            phi_B = phi_B + 1 - phi_A
            t_B_to_A = t_B_to_A - (1 - phi_A)
            t_A_to_B = delta
            t_present = t_present + 1 - phi_A
            num_spikes_A += 1
            t_spikes_A.append(t_present)
            phi_A = 0.
            done = True

        if T_0 == 1 - phi_B and not done:
            phi_A = phi_A + 1 - phi_B
            t_A_to_B = t_A_to_B - (1 - phi_B)
            t_B_to_A = delta
            t_present = t_present + 1 - phi_B
            num_spikes_B += 1
            t_spikes_B.append(t_present)
            phi_B = 0.
            done = True

        if T_0 == t_A_to_B and not done:
            phi_B = two_pulse_f(phi_B + t_A_to_B, epsilon)
            phi_A = phi_A + t_A_to_B
            t_B_to_A = t_B_to_A - t_A_to_B
            t_present = t_present + t_A_to_B
            t_A_to_B = np.inf
            done = True

        if T_0 == t_B_to_A and not done:
            phi_A = two_pulse_f(phi_A + t_B_to_A, epsilon)
            phi_B = phi_B + t_B_to_A
            t_A_to_B = t_A_to_B - t_B_to_A
            t_present = t_present + t_B_to_A
            t_B_to_A = np.inf
            done = True

    return np.array(t_spikes_A), np.array(t_spikes_B)


def simulate_two_delayed_pulse_coupled_osc(t_final=20.):
    t_spikes_A_1, t_spikes_B_1 = simulate_two_delayed_pulse_pair(delta=0.1, t_final=t_final)
    t_spikes_A_2, t_spikes_B_2 = simulate_two_delayed_pulse_pair(delta=0.7, t_final=t_final)
    return t_spikes_A_1, t_spikes_B_1, t_spikes_A_2, t_spikes_B_2


def plot_two_delayed_pulse_coupled_osc(t_spikes_A_1, t_spikes_B_1, t_spikes_A_2, t_spikes_B_2, t_final=20.):
    fig, axes = plt.subplots(2, 1, figsize=(8, 6))

    axes[0].plot(t_spikes_A_1, np.ones(len(t_spikes_A_1)), '.r', markersize=15)
    axes[0].plot(t_spikes_B_1, 2 * np.ones(len(t_spikes_B_1)), '.g', markersize=15)
    axes[0].axis([t_final - 10, t_final, 0, 3])
    axes[0].set_yticks([])
    axes[0].set_title(r'$\delta=0.1$:')

    axes[1].plot(t_spikes_A_2, np.ones(len(t_spikes_A_2)), '.r', markersize=15)
    axes[1].plot(t_spikes_B_2, 2 * np.ones(len(t_spikes_B_2)), '.g', markersize=15)
    axes[1].axis([t_final - 10, t_final, 0, 3])
    axes[1].set_yticks([])
    axes[1].set_xlabel('$t$ [units of $T$]')
    axes[1].set_title(r'$\delta=0.7$:')

    plt.tight_layout()
    plt.show()


In [ ]:
t_spikes_A_1, t_spikes_B_1, t_spikes_A_2, t_spikes_B_2 = simulate_two_delayed_pulse_coupled_osc()
plot_two_delayed_pulse_coupled_osc(t_spikes_A_1, t_spikes_B_1, t_spikes_A_2, t_spikes_B_2)


## Three Delayed Pulse-Coupled Oscillators

Extending the same event-driven scheme to $N=3$ all-to-all pulse-coupled
oscillators: each spike sends $N-1$ signals, one to every other oscillator,
each arriving $\delta$ time units later and nudging the receiving
oscillator's phase. `simulate_three_delayed_pulse_coupled_osc` compares a
smaller delay ($\delta=0.45$) against a larger one ($\delta=0.55$); the
larger delay brings the three oscillators much closer to synchrony.


In [ ]:
def three_pulse_g(phi, scale):
    return phi * (1 - phi) / 3 / scale


def simulate_three_delayed_pulse_pair(delta, phi0, t_final=200., scale=1., N=3):
    '''event-driven simulation of N all-to-all pulse-coupled oscillators
    with a uniform transmission delay delta (in units of the intrinsic
    period). Each spike sends N-1 signals (one to every other
    oscillator), each arriving delta time units later and nudging the
    receiving oscillator's phase by g(phi)/scale.'''
    t_0 = 0.
    phi = phi0.copy()
    num_spikes = 0
    t_spikes = []
    i_spikes = []

    M = 0
    t = np.zeros(N * N)
    k_to = np.zeros(N * N, dtype=int)
    k_from = np.zeros(N * N, dtype=int)

    while t_0 < t_final:
        next_spike = np.min(1 - phi)
        next_signal = np.inf if M == 0 else np.min(t[:M])
        next_event = min(next_spike, next_signal)

        if next_event == next_signal:
            j0 = np.max(np.where(t[:M] == next_signal)[0])
            phi = phi + next_signal
            t_0 = t_0 + next_signal
            t = t - next_signal
            target = k_to[j0]
            phi[target] = phi[target] + three_pulse_g(phi[target], scale)

            t[j0:M - 1] = t[j0 + 1:M]
            k_to[j0:M - 1] = k_to[j0 + 1:M]
            k_from[j0:M - 1] = k_from[j0 + 1:M]
            M -= 1

        else:
            i0 = np.max(np.where(1 - phi == next_spike)[0])
            phi = phi + next_spike
            t_0 = t_0 + next_spike
            t[:M] = t[:M] - next_spike
            phi[i0] = 0.

            num_spikes += 1
            t_spikes.append(t_0)
            i_spikes.append(i0)

            targets = [j for j in range(N) if j != i0]
            for l, tgt in enumerate(targets):
                k_from[M + l] = i0
                k_to[M + l] = tgt
                t[M + l] = delta
            M += N - 1

    return np.array(t_spikes), np.array(i_spikes)


def simulate_three_delayed_pulse_coupled_osc(t_final=200., seed=63806, N=3):
    # MATLAB's rng('default'); rng(63806) cannot be bit-reproduced by
    # NumPy, so this uses its own seed; the port is verified
    # statistically/visually rather than against exact MATLAB spike times.
    rng = np.random.default_rng(seed)

    phi0_1 = 0.9 + rng.random(N) * 0.1
    t_spikes_1, i_spikes_1 = simulate_three_delayed_pulse_pair(
        delta=0.45, phi0=phi0_1, t_final=t_final, scale=N - 1)

    phi0_2 = 0.9 + rng.random(N) * 0.1
    t_spikes_2, i_spikes_2 = simulate_three_delayed_pulse_pair(
        delta=0.55, phi0=phi0_2, t_final=t_final, scale=1.)

    return t_spikes_1, i_spikes_1, t_spikes_2, i_spikes_2


def plot_three_delayed_pulse_coupled_osc(t_spikes_1, i_spikes_1, t_spikes_2, i_spikes_2, t_final=200., N=3):
    fig, axes = plt.subplots(2, 1, figsize=(8, 6))

    axes[0].plot(t_spikes_1, i_spikes_1 + 1, '.k', markersize=15)
    axes[0].axis([t_final - 10, t_final, 0, N + 1])
    axes[0].set_yticks([1, 2, 3])
    axes[0].set_title(r'$\delta=0.45$:')

    axes[1].plot(t_spikes_2, i_spikes_2 + 1, '.k', markersize=15)
    axes[1].axis([t_final - 10, t_final, 0, N + 1])
    axes[1].set_yticks([1, 2, 3])
    axes[1].set_xlabel('$t$ [units of $T$]')
    axes[1].set_title(r'$\delta=0.55$:')

    plt.tight_layout()
    plt.show()


In [ ]:
t_spikes_1, i_spikes_1, t_spikes_2, i_spikes_2 = simulate_three_delayed_pulse_coupled_osc()
plot_three_delayed_pulse_coupled_osc(t_spikes_1, i_spikes_1, t_spikes_2, i_spikes_2)


## Two Delayed, Pulse-Coupled Theta Neurons

The same delayed-pulse locking realized with continuous theta neurons
(Chapter 8) instead of an abstract phase map: whenever one neuron spikes,
its effect on the other arrives $\delta T$ time units later and jumps the
receiving neuron's phase by $\theta\to2\arctan(\tan(\theta/2)+2\,d v)$.
`simulate_two_theta_neurons_grid` sweeps a $9\times9$ grid of coupling
strengths $\epsilon$ and delays $\delta$, classifying each pair as
synchronized or unsynchronized, and compares the result with the
theoretical boundary $\delta=\tfrac12-\tfrac1\pi\arctan(\epsilon/2)$.


In [ ]:
def theta_neuron_inc(theta, tau_m=2., I=0.14):
    return -np.cos(theta) / tau_m + 2 * I * (1 + np.cos(theta))


def simulate_two_theta_neurons_pair(delta, epsilon, tau_m=2., I=0.14, t_final=10000., dt=0.01):
    '''two theta neurons A, B coupled by delayed pulses: whenever B (A)
    spikes, its effect on A (B) arrives delta*T time units later, and
    jumps A's (B's) theta by the phase-response map
    theta -> 2*atan(tan(theta/2) + 2*dv).

    Faithfully reproduces the matlab source's bug in the "no delayed
    input this step" branch for theta_B, which advances theta_B using
    theta_A_inc (leftover from theta_A's update earlier in the same
    step) instead of computing/using its own increment.'''
    T = np.pi * tau_m / np.sqrt(tau_m * I - 0.25)
    m_steps = round(t_final / dt)
    dt05 = dt / 2
    dv = np.sqrt(tau_m * I - 0.25) * epsilon

    theta_A = 0.
    theta_B = np.pi / 6
    t_spikes_A = []
    t_spikes_B = []

    # monotonically-advancing pointers into the other neuron's spike
    # list: since delta*T is a fixed offset and spike times only
    # increase, the next not-yet-delivered spike is found in O(1)
    # amortized instead of re-scanning the whole list every step.
    ptr_A = 0  # next unconsumed entry in t_spikes_B (deliveries to A)
    ptr_B = 0  # next unconsumed entry in t_spikes_A (deliveries to B)

    for k in range(1, m_steps + 1):
        t_lo, t_hi = (k - 1) * dt, k * dt

        theta_A_inc = None
        if ptr_A < len(t_spikes_B) and t_lo < t_spikes_B[ptr_A] + delta * T <= t_hi:
            t_0 = t_spikes_B[ptr_A] + delta * T
            ptr_A += 1
            dt_1 = t_0 - t_lo
            dt_105 = dt_1 / 2
            theta_A_inc = theta_neuron_inc(theta_A, tau_m, I)
            theta_A_tmp = theta_A + dt_105 * theta_A_inc
            theta_A_inc = theta_neuron_inc(theta_A_tmp, tau_m, I)
            t_A = theta_A + dt_1 * theta_A_inc
            t_A = 2 * np.arctan(np.tan(t_A / 2) + 2 * dv)
            dt_2 = t_hi - t_0
            dt_205 = dt_2 / 2
            theta_A_inc = theta_neuron_inc(t_A, tau_m, I)
            theta_A_tmp = t_A + dt_205 * theta_A_inc
            theta_A_inc = theta_neuron_inc(theta_A_tmp, tau_m, I)
            theta_A_next = t_A + dt_2 * theta_A_inc
        else:
            theta_A_inc = theta_neuron_inc(theta_A, tau_m, I)
            theta_A_tmp = theta_A + dt05 * theta_A_inc
            theta_A_inc = theta_neuron_inc(theta_A_tmp, tau_m, I)
            theta_A_next = theta_A + dt * theta_A_inc

        if ptr_B < len(t_spikes_A) and t_lo < t_spikes_A[ptr_B] + delta * T <= t_hi:
            t_0 = t_spikes_A[ptr_B] + delta * T
            ptr_B += 1
            dt_1 = t_0 - t_lo
            dt_105 = dt_1 / 2
            theta_B_inc = theta_neuron_inc(theta_B, tau_m, I)
            theta_B_tmp = theta_B + dt_105 * theta_B_inc
            theta_B_inc = theta_neuron_inc(theta_B_tmp, tau_m, I)
            t_B = theta_B + dt_1 * theta_B_inc
            t_B = 2 * np.arctan(np.tan(t_B / 2) + 2 * dv)
            dt_2 = t_hi - t_0
            dt_205 = dt_2 / 2
            theta_B_inc = theta_neuron_inc(t_B, tau_m, I)
            theta_B_tmp = t_B + dt_205 * theta_B_inc
            theta_B_inc = theta_neuron_inc(theta_B_tmp, tau_m, I)
            theta_B_next = t_B + dt_2 * theta_B_inc
        else:
            # faithful port of the matlab bug: uses theta_A_inc (left
            # over from the A update above), not a freshly computed
            # theta_B_inc.
            theta_B_next = theta_B + dt * theta_A_inc

        if theta_A_next > np.pi:
            tt = (t_lo * (theta_A_next - np.pi) + t_hi * (np.pi - theta_A)) / (theta_A_next - theta_A)
            t_spikes_A.append(tt)
            theta_A_next -= 2 * np.pi

        if theta_B_next > np.pi:
            tt = (t_lo * (theta_B_next - np.pi) + t_hi * (np.pi - theta_B)) / (theta_B_next - theta_B)
            t_spikes_B.append(tt)
            theta_B_next -= 2 * np.pi

        theta_A, theta_B = theta_A_next, theta_B_next

    return np.array(t_spikes_A), np.array(t_spikes_B)


def sync_measure(t_spikes_A, t_spikes_B):
    d = np.array([np.min(np.abs(ta - t_spikes_B)) for ta in t_spikes_A])
    return d[len(t_spikes_A) - 2]


def simulate_two_theta_neurons_grid(N=10):
    epsilon_vec = np.arange(1, 1001) / 1000 * 5
    delta_vec = 1 / 2 - 1 / np.pi * np.arctan(epsilon_vec / 2)

    grid_points = []  # (epsilon, delta, is_synced)
    for i in range(1, N):
        for j in range(1, N):
            delta = j / N
            epsilon = i / N * 5
            t_spikes_A, t_spikes_B = simulate_two_theta_neurons_pair(delta, epsilon)
            synced = sync_measure(t_spikes_A, t_spikes_B) < 1e-2
            grid_points.append((epsilon, delta, synced))

    return epsilon_vec, delta_vec, grid_points


def plot_two_theta_neurons_grid(epsilon_vec, delta_vec, grid_points):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(epsilon_vec, delta_vec, '-k', linewidth=5)
    ax.set_xlabel(r'$\epsilon$')
    ax.set_ylabel(r'$\delta$')
    ax.axis([0, 5, 0, 1])
    ax.set_box_aspect(1)

    poly_x = np.concatenate(([0.], epsilon_vec, [5.], epsilon_vec[::-1], [0.]))
    poly_y = np.concatenate(([0.5], delta_vec, [1.], np.ones(1000), [1.]))
    ax.add_patch(Polygon(np.column_stack([poly_x, poly_y]), closed=True,
                          facecolor='y', alpha=0.2))

    for epsilon, delta, synced in grid_points:
        if synced:
            ax.plot(epsilon, delta, '.r', markersize=15)
        else:
            ax.plot(epsilon, delta, '.b', markersize=15)

    plt.tight_layout()
    plt.show()


In [ ]:
# ~90s: 81 grid points x 1e6-step simulations of two delay-coupled theta neurons each.
epsilon_vec, delta_vec, grid_points = simulate_two_theta_neurons_grid()
plot_two_theta_neurons_grid(epsilon_vec, delta_vec, grid_points)
